# 08 — Inference Demo

Демонстрация end-to-end pipeline предсказания вектора CVSS v4.0 на основе обученной модели `models/final_model.pt`.

Состав ноутбука:
1. Загрузка `VulnerabilityPredictor`.
2. Прогон на ~15 примерах из `data/processed/test.parquet` с известными ответами.
3. Сравнительная таблица и подсчёт точности по метрикам.
4. Несколько «ручных» примеров на английском и русском.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option('display.max_colwidth', 80)

from src.inference import VulnerabilityPredictor

C:\Users\Артём\Desktop\diplom\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Загрузка модели

In [2]:
predictor = VulnerabilityPredictor(
    model_path=str(ROOT / 'models' / 'final_model.pt'),
    config_path=str(ROOT / 'configs' / 'train.yaml'),
    cwe_vocab_path=str(ROOT / 'data' / 'processed' / 'cwe_vocab.json'),
    device='auto',
    confidence_threshold=0.7,
)
print('device:', predictor.device)
print('classes:', predictor.model.metric_classes)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11941.55it/s]


[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


device: cpu
classes: {'AV': 4, 'AC': 2, 'AT': 2, 'PR': 3, 'UI': 3, 'VC': 3, 'VI': 3, 'VA': 3, 'SC': 3, 'SI': 3, 'SA': 3, 'E': 3}


## 2. Прогон на 15 примерах из test.parquet

In [3]:
test_df = pd.read_parquet(ROOT / 'data' / 'processed' / 'test.parquet')
test_v4 = test_df[test_df['cvss_v4_vector'].notna()].reset_index(drop=True)
sample = test_v4.sample(15, random_state=42).reset_index(drop=True)
print(f'Доступно с CVSS v4: {len(test_v4)} | в выборке: {len(sample)}')
sample[['cve_id', 'cwe_id', 'cvss_v4_vector']].head()

Доступно с CVSS v4: 972 | в выборке: 15


,cve_id,cwe_id,cvss_v4_vector
0,CVE-2025-4181,CWE-119,AV:N/AC:L/AT:N/PR:N/UI:N/VC:L/VI:L/VA:L/SC:N/SI:N/SA:N
1,CVE-2024-10159,CWE-89,CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:L/VI:L/VA:L/SC:N/SI:N/SA:N/E:X/CR:X/IR:...
2,CVE-2024-52567,CWE-125,AV:L/AC:H/AT:N/PR:N/UI:P/VC:H/VI:H/VA:H/SC:N/SI:N/SA:N
3,CVE-2024-4914,CWE-89,CVSS:4.0/AV:N/AC:L/AT:N/PR:L/UI:N/VC:L/VI:L/VA:L/SC:N/SI:N/SA:N/E:X/CR:X/IR:...
4,CVE-2021-25663,CWE-835,CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:N/VI:N/VA:H/SC:N/SI:N/SA:N/E:X/CR:X/IR:...


In [4]:
items = []
for _, row in sample.iterrows():
    items.append({
        'description': row['d_en'] if pd.notna(row['d_en']) else None,
        'description_ru': row['d_ru'] if pd.notna(row['d_ru']) else None,
        'cwe_id': row['cwe_id'],
        'epss': row['epss'] if pd.notna(row['epss']) else None,
        'kev': row['kev'] if pd.notna(row['kev']) else None,
        'exploit': row['exploit'] if pd.notna(row['exploit']) else None,
    })

results = predictor.predict_batch(items, batch_size=8)
print(f'Получено {len(results)} предсказаний')

Получено 15 предсказаний


## 3. Сравнительная таблица: true vs predicted

In [5]:
from src.cvss_calculator import CVSSCalculator
calc = CVSSCalculator()

def trim(vec: str) -> str:
    return '/'.join(p for p in vec.split('/') if ':X' not in p)

def true_score(vec: str) -> float:
    try:
        metrics = calc.parse_vector_string(vec)
        metrics.setdefault('E', 'A')
        score, _, _ = calc.calculate({k: v for k, v in metrics.items() if v != 'X'})
        return score
    except Exception:
        return float('nan')

def pick_text(row):
    if pd.notna(row.get('d_en')) and isinstance(row['d_en'], str):
        return row['d_en']
    if pd.notna(row.get('d_ru')) and isinstance(row['d_ru'], str):
        return row['d_ru']
    return ''

rows = []
for (_, row), result in zip(sample.iterrows(), results):
    true_vec = row['cvss_v4_vector']
    pred_vec = result['vector']
    true_metrics = calc.parse_vector_string(true_vec)
    pred_metrics = result['metrics']
    base = ['AV', 'AC', 'AT', 'PR', 'UI', 'VC', 'VI', 'VA', 'SC', 'SI', 'SA']
    matched = sum(1 for m in base if true_metrics.get(m) == pred_metrics.get(m))
    rows.append({
        'cve_id': row['cve_id'],
        'description': pick_text(row)[:60],
        'true_vector': trim(true_vec),
        'pred_vector': trim(pred_vec),
        'true_score': true_score(true_vec),
        'pred_score': result['score'],
        'match_11': f'{matched}/11',
    })

comparison = pd.DataFrame(rows)
comparison

,cve_id,description,true_vector,pred_vector,true_score,pred_score,match_11
0,CVE-2025-4181,Уязвимость FTP-сервера PCMan FTP Server связана с выходом оп,AV:N/AC:L/AT:N/PR:N/UI:N/VC:L/VI:L/VA:L/SC:N/SI:N/SA:N,CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:L/VI:L/VA:L/SC:N/SI:N/SA:N/E:A,6.9,6.9,11/11
1,CVE-2024-10159,A vulnerability classified as critical was found in PHPGuruk,CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:L/VI:L/VA:L/SC:N/SI:N/SA:N,CVSS:4.0/AV:N/AC:L/AT:N/PR:L/UI:N/VC:L/VI:L/VA:L/SC:N/SI:N/SA:N/E:A,6.9,5.3,10/11
2,CVE-2024-52567,A vulnerability has been identified in Teamcenter Visualizat,AV:L/AC:H/AT:N/PR:N/UI:P/VC:H/VI:H/VA:H/SC:N/SI:N/SA:N,CVSS:4.0/AV:L/AC:H/AT:N/PR:N/UI:P/VC:H/VI:H/VA:H/SC:N/SI:N/SA:N/E:A,7.3,7.3,11/11
3,CVE-2024-4914,"A vulnerability, which was classified as critical, has been",CVSS:4.0/AV:N/AC:L/AT:N/PR:L/UI:N/VC:L/VI:L/VA:L/SC:N/SI:N/SA:N,CVSS:4.0/AV:N/AC:L/AT:N/PR:L/UI:N/VC:L/VI:L/VA:L/SC:N/SI:N/SA:N/E:A,5.3,5.3,11/11
4,CVE-2021-25663,A vulnerability has been identified in Capital Embedded AR C,CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:N/VI:N/VA:H/SC:N/SI:N/SA:N,CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:N/VI:N/VA:H/SC:N/SI:N/SA:N/E:A,8.7,8.7,11/11
5,CVE-2025-50193,Уязвимость системы электронного обучения и управления контен,AV:N/AC:L/AT:N/PR:H/UI:N/VC:L/VI:H/VA:H/SC:N/SI:N/SA:N,CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:H/VI:L/VA:H/SC:N/SI:N/SA:N/E:A,7.1,8.8,8/11
6,CVE-2025-27127,Уязвимость микропрограммного обеспечения сервера TIA Project,AV:N/AC:L/AT:N/PR:L/UI:N/VC:N/VI:N/VA:L/SC:N/SI:N/SA:N,CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:N/VI:N/VA:H/SC:N/SI:N/SA:N/E:A,5.3,8.7,9/11
7,CVE-2024-9282,A vulnerability was found in bg5sbk MiniCMS 1.11. It has bee,CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:N/VI:L/VA:N/SC:N/SI:N/SA:N,CVSS:4.0/AV:N/AC:L/AT:N/PR:L/UI:N/VC:L/VI:N/VA:L/SC:N/SI:N/SA:N/E:A,6.9,5.3,7/11
8,CVE-2025-29913,Уязвимость функции Crypto_TC_Prep_AAD библиотеки CryptoLib с,AV:N/AC:L/AT:N/PR:N/UI:N/VC:H/VI:H/VA:H/SC:N/SI:N/SA:N/E:P,CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:N/VI:N/VA:H/SC:N/SI:N/SA:N/E:U,8.9,6.6,9/11
9,CVE-2024-39549,A Missing Release of Memory after Effective Lifetime vulnera,AV:N/AC:L/AT:N/PR:N/UI:N/VC:N/VI:N/VA:H/SC:N/SI:N/SA:L/R:U,CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:N/VI:N/VA:H/SC:N/SI:N/SA:L/E:A,8.7,8.7,11/11


In [6]:
matched_total = sum(int(r['match_11'].split('/')[0]) for r in rows)
total_metrics = 11 * len(rows)
print(f'Суммарно угадано: {matched_total}/{total_metrics} = {matched_total/total_metrics:.1%}')
score_diff = (comparison['pred_score'] - comparison['true_score']).abs().mean()
print(f'Средняя |Δ score|: {score_diff:.2f}')
exact_vector = (comparison['true_vector'] == comparison['pred_vector']).sum()
print(f'Полное совпадение вектора: {exact_vector}/{len(comparison)}')

Суммарно угадано: 141/165 = 85.5%
Средняя |Δ score|: 1.24
Полное совпадение вектора: 0/15


## 4. Ручные примеры

In [7]:
manual_cases = [
    {
        'name': 'SQL injection / auth bypass',
        'description': 'SQL injection allows authentication bypass via login form',
        'cwe_id': 'CWE-89',
    },
    {
        'name': 'Buffer overflow → RCE',
        'description': 'Buffer overflow in image parser leading to remote code execution',
        'cwe_id': 'CWE-120',
    },
    {
        'name': 'Переполнение буфера в драйвере ядра (ru)',
        'description': 'Уязвимость переполнения буфера в драйвере ядра позволяет локальному пользователю выполнить произвольный код с правами SYSTEM',
        'cwe_id': 'CWE-787',
    },
    {
        'name': 'Stored XSS',
        'description': 'Authenticated stored XSS vulnerability in admin dashboard allows attacker to execute scripts in victim browser',
        'cwe_id': 'CWE-79',
    },
    {
        'name': 'Path traversal',
        'description': 'Path traversal in HTTP file server allows unauthenticated attacker to read arbitrary files',
        'cwe_id': 'CWE-22',
    },
]

manual_results = predictor.predict_batch(
    [{'description': c['description'], 'cwe_id': c['cwe_id']} for c in manual_cases]
)

manual_rows = []
for case, res in zip(manual_cases, manual_results):
    manual_rows.append({
        'name': case['name'],
        'cwe': case['cwe_id'],
        'vector': res['vector'].replace('CVSS:4.0/', ''),
        'score': res['score'],
        'severity': res['severity'],
        'low_conf': ','.join(res['low_confidence_metrics']) or '-',
    })
pd.DataFrame(manual_rows)

,name,cwe,vector,score,severity,low_conf
0,SQL injection / auth bypass,CWE-89,AV:N/AC:L/AT:N/PR:N/UI:N/VC:L/VI:L/VA:L/SC:N/SI:N/SA:N/E:A,6.9,Medium,-
1,Buffer overflow → RCE,CWE-120,AV:L/AC:H/AT:N/PR:N/UI:P/VC:H/VI:H/VA:H/SC:N/SI:N/SA:N/E:A,7.3,High,"AT,PR,UI,VC,SI,SA"
2,Переполнение буфера в драйвере ядра (ru),CWE-787,AV:L/AC:L/AT:N/PR:N/UI:P/VC:H/VI:H/VA:H/SC:N/SI:N/SA:N/E:A,8.5,High,"AC,UI"
3,Stored XSS,CWE-79,AV:L/AC:L/AT:N/PR:N/UI:P/VC:H/VI:H/VA:H/SC:N/SI:N/SA:N/E:A,8.5,High,"PR,UI"
4,Path traversal,CWE-22,AV:N/AC:L/AT:N/PR:N/UI:N/VC:H/VI:L/VA:N/SC:N/SI:N/SA:N/E:A,8.8,High,"AV,VC,VI,VA,SI"


## Вывод

Pipeline корректно работает на реальных примерах из тестовой выборки и на ручных описаниях.
Уверенные метрики (`confidence ≥ 0.7`) — AV, AC, AT, UI, SC, SI, SA, E.
Низкоуверенные обычно — PR, VC, VI, VA, что соответствует общим выводам из notebook 04 (impact-метрики — самые трудные).